# 02 BIDMC Canonical Window Export

Build canonical BIDMC 10s windows (5s stride) and export engineered features for the grouped baseline model.

Inputs from: `outputs/bidmc_dataset_file_coverage.csv` and `outputs/bidmc_audit.csv` (from notebooks 00 and 01).
Outputs to: `outputs/bidmc_window_features.csv` and refreshed `outputs/bidmc_audit.csv`.
Next: notebook 03 for grouped logistic-regression baseline evaluation.

In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
from scipy import signal, stats

REQUIRED_FILE_KINDS = ('Signals', 'Numerics', 'Breaths', 'Fix')

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'notebooks').is_dir() and (candidate / 'outputs').is_dir():
            return candidate
    raise RuntimeError('Could not find project root containing notebooks/ and outputs/')

def resolve_bidmc_dir(project_root: Path) -> Path:
    env_path = os.getenv('BIDMC_DATASET_DIR', '').strip()
    if env_path:
        p = Path(env_path).expanduser().resolve()
        if p.is_dir():
            return p
    default_path = (project_root / 'bidmc_csv').resolve()
    if default_path.is_dir():
        return default_path
    raise FileNotFoundError('BIDMC dataset directory not found. Set BIDMC_DATASET_DIR or place data at <repo>/bidmc_csv')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
OUTPUTS_DIR = (PROJECT_ROOT / 'outputs').resolve()
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

for required in ['bidmc_dataset_file_coverage.csv', 'bidmc_audit.csv']:
    required_path = OUTPUTS_DIR / required
    if not required_path.is_file():
        raise FileNotFoundError(f'Missing required audit artifact: {required_path}. Run notebooks 00 and 01 first.')

DATASET_DIR = resolve_bidmc_dir(PROJECT_ROOT)
AUDIT_CSV = OUTPUTS_DIR / 'bidmc_audit.csv'
FEATURE_CSV = OUTPUTS_DIR / 'bidmc_window_features.csv'

print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset dir:  {DATASET_DIR}')
print(f'Outputs dir:  {OUTPUTS_DIR}')


## Environment And Core Settings

Set project paths and core preprocessing thresholds used in the rest of the notebook.

## Dataset Validation And Recording Discovery

Validate required BIDMC file groups and build a per-recording manifest for downstream processing.

In [ ]:
# Discover all BIDMC recordings and validate that each has Signals, Numerics, Breaths, and Fix files.
# BIDMC recordings are organized as sets of 4 file types, all sharing a common recording ID prefix.
all_files = sorted([p for p in DATASET_DIR.iterdir() if p.is_file()])
recording_file_map: dict[str, dict[str, Path]] = {}

# ===== BUILD RECORDING FILE MAP =====
# Parse each file stem to extract recording ID and file kind.
# Example: "bidmc_01_Signals.csv" → recording_id="bidmc_01", file_kind="Signals"
for path in all_files:
    parts = path.stem.split("_")
    # Skip files that don't match expected pattern
    if len(parts) < 3 or parts[0] != "bidmc":
        continue

    # Reconstruct recording ID (e.g., "bidmc_01") and extract file kind (e.g., "Signals")
    recording_id = "_".join(parts[:2])
    file_kind = parts[2]
    
    # Initialize dict for this recording if new
    if recording_id not in recording_file_map:
        recording_file_map[recording_id] = {}
    
    # Map this file by its kind
    recording_file_map[recording_id][file_kind] = path

# ===== VALIDATE COMPLETENESS =====
# Check that every recording has all 4 required file kinds: Signals, Numerics, Breaths, Fix
# If any recording is missing files, report error with examples.
missing_by_recording: dict[str, list[str]] = {}
for recording_id in sorted(recording_file_map):
    file_group = recording_file_map[recording_id]
    # Find which required kinds are absent from this recording
    missing = [kind for kind in REQUIRED_FILE_KINDS if kind not in file_group]
    if missing:
        missing_by_recording[recording_id] = missing

if missing_by_recording:
    preview = {k: v for k, v in list(missing_by_recording.items())[:5]}
    raise ValueError(f"Some recordings are missing required files. Example: {preview}")

# ===== BUILD RECORDINGS MANIFEST =====
# Create a structured DataFrame mapping each recording ID to its file paths.
# This manifest is used to load data for each recording in downstream cells.
recording_ids = sorted(recording_file_map.keys())
recordings_manifest = pd.DataFrame(
    [
        {
            "recording_id": rid,
            "signals_path": str(recording_file_map[rid]["Signals"]),
            "numerics_path": str(recording_file_map[rid]["Numerics"]),
            "breaths_path": str(recording_file_map[rid]["Breaths"]),
            "fix_path": str(recording_file_map[rid]["Fix"]),
        }
        for rid in recording_ids
    ]
)

print(f"Discovered recordings: {len(recording_ids)}")
recordings_manifest.head()

### Step 1: Build Dataset Audit Table

Load each recording's signals and numerics files, validate required columns, and summarize row counts plus SpO2 quality checks into an audit DataFrame.

In [ ]:
# Build dataset audit rows directly from each recording's Signals and Numerics files.
audit_rows: list[dict[str, float | int | str | None]] = []

for recording_id in recording_ids:
    signals_path = recording_file_map[recording_id]["Signals"]
    numerics_path = recording_file_map[recording_id]["Numerics"]

    signals_df = pd.read_csv(signals_path, skipinitialspace=True)
    numerics_df = pd.read_csv(numerics_path, skipinitialspace=True)

    for required_col in ["Time [s]", "PLETH"]:
        if required_col not in signals_df.columns:
            raise ValueError(f"Missing required column {required_col} in {signals_path}")

    for required_col in ["Time [s]", "HR", "PULSE", "RESP", "SpO2"]:
        if required_col not in numerics_df.columns:
            raise ValueError(f"Missing required column {required_col} in {numerics_path}")

    spo2_values = pd.to_numeric(numerics_df["SpO2"], errors="coerce")
    observed_spo2 = spo2_values.dropna()

    audit_rows.append(
        {
            "recording_id": recording_id,
            "signal_row_count": int(len(signals_df)),
            "numerics_row_count": int(len(numerics_df)),
            "spo2_missing_count": int(spo2_values.isna().sum()),
            "spo2_below_90_count": int((spo2_values < 90).sum(skipna=True)),
            "spo2_89_to_91_count": int(((spo2_values >= 89) & (spo2_values <= 91)).sum(skipna=True)),
            "spo2_min": float(observed_spo2.min()) if len(observed_spo2) else None,
            "spo2_max": float(observed_spo2.max()) if len(observed_spo2) else None,
        }
    )

audit_df = pd.DataFrame(audit_rows).sort_values("recording_id").reset_index(drop=True)
audit_df.head()

## Pleth Preprocessing, Window Labeling, And Feature Extraction

Resample and band-pass each PLETH signal, generate labeled windows from median SpO2 as the primary target, and retain minimum SpO2 as a secondary sensitivity signal.

### Step 1: Define Preprocessing Parameters

These thresholds control resampling rate, filter bounds, window dimensions, and SpO2 classification cutoffs.

In [ ]:
# Preprocessing and windowing parameters
target_rate_hz = 50.0  # Target resampling rate
low_cut_hz = 0.5  # Band-pass filter lower cutoff
high_cut_hz = 8.0  # Band-pass filter upper cutoff
filter_order = 4  # Butterworth filter order

# Window parameters
window_seconds = 10.0  # Duration of each window
stride_seconds = 5.0  # Overlap between consecutive windows

# SpO2 labeling thresholds
low_spo2_threshold = 90.0  # Hypoxemia threshold
borderline_margin = 1.0  # Margin around threshold (±1%)
drop_borderline = True  # Drop samples in borderline zone

# Derived sample counts
window_size_samples = int(round(window_seconds * target_rate_hz))
stride_size_samples = int(round(stride_seconds * target_rate_hz))
borderline_lower = low_spo2_threshold - borderline_margin
borderline_upper = low_spo2_threshold + borderline_margin

print(f"Window: {window_size_samples} samples ({window_seconds}s @ {target_rate_hz} Hz)")
print(f"Stride: {stride_size_samples} samples ({stride_seconds}s overlap)")
print(f"SpO2 thresholds: <{low_spo2_threshold}=Hypoxemia, {borderline_lower}-{borderline_upper}=Borderline (drop={drop_borderline})")

### Step 2: Signal Processing Pipeline Overview

For each recording, the pipeline:
1. **Load**: Read raw PLETH signal and SpO2 numerics
2. **Clean**: Remove rows with missing time or signal values
3. **Resample**: Resample raw PLETH to uniform 50 Hz rate
4. **Filter**: Apply band-pass filter (0.5–8 Hz) to remove noise and trends
5. **Window**: Slide a 10-second window with 5-second stride
6. **Label**: Assign binary label (hypoxemia vs. normal) based on median SpO2 in window
7. **Extract**: Compute 14 features (statistical, spectral, temporal) for each window

In [ ]:
# === INITIALIZE COLLECTION ===
feature_rows: list[dict[str, float | int | str]] = []
example_windows_count = 0


def compute_window_features(values: np.ndarray, sample_rate_hz: float) -> dict[str, float]:
    """
    Compute 14 temporal, statistical, and spectral features from a PLETH signal window.
    
    This function is the primary feature extraction pipeline. It processes a clean resampled
    and band-pass filtered PLETH window and returns a dictionary of computed features used
    for downstream classification (hypoxemia vs. normal).
    
    Parameters:
    -----------
    values : np.ndarray
        1-D array of resampled PLETH samples (typically 500 samples for a 10-second window at 50 Hz).
        Should be finite (no NaN/inf). Assumed to already be band-pass filtered.
    sample_rate_hz : float
        Sampling rate of the signal in Hz (typically 50.0). Used for spectral analysis (Welch PSD).
    
    Returns:
    --------
    dict[str, float]
        Dictionary with 14 feature keys:
        - 'mean', 'std', 'minimum', 'maximum', 'value_range': Basic statistics
        - 'rms': Root mean square
        - 'skewness', 'kurtosis': Higher-order statistics (set to 0.0 if non-finite)
        - 'energy': Sum of squared samples
        - 'derivative_std': Std dev of first differences (motion/variability indicator)
        - 'dominant_frequency_hz': Peak frequency in power spectrum
        - 'bandpower_0_5_2_hz': Power in respiration band (0.5–2 Hz)
        - 'bandpower_2_5_hz': Power in higher band (2–5 Hz)
        - 'bandpower_ratio_0_5_2_to_2_5': Ratio of resp band to higher band
    
    Edge Cases:
    -----------
    - Non-finite skewness/kurtosis: Set to 0.0
    - Empty or single-sample values: Returns dict with 0.0 for spectral features
    - Insufficient frequency bins in Welch analysis: Returns 0.0 for dominant_frequency_hz
    - Zero power in higher band: bandpower_ratio set to 0.0
    """
    # ===== TEMPORAL AND STATISTICAL FEATURES =====
    mean_value = float(np.mean(values))
    std_value = float(np.std(values))
    minimum_value = float(np.min(values))
    maximum_value = float(np.max(values))
    value_range = maximum_value - minimum_value
    rms_value = float(np.sqrt(np.mean(np.square(values))))

    # Skewness and kurtosis may be non-finite if signal length is too short;
    # clamp to 0.0 for stability in downstream models.
    skewness_value = float(stats.skew(values, bias=False))
    if not np.isfinite(skewness_value):
        skewness_value = 0.0

    kurtosis_value = float(stats.kurtosis(values, fisher=True, bias=False))
    if not np.isfinite(kurtosis_value):
        kurtosis_value = 0.0

    # Energy (sum of squares) and derivative variability (first-order differences).
    energy_value = float(np.sum(np.square(values)))
    derivative_std_value = float(np.std(np.diff(values))) if len(values) > 1 else 0.0

    # ===== SPECTRAL FEATURES (WELCH PERIODOGRAM) =====
    # Compute power spectral density to extract frequency-domain features.
    frequencies, power = signal.welch(
        values,
        fs=sample_rate_hz,
        nperseg=min(256, len(values)),
    )

    if len(frequencies) <= 1 or len(power) <= 1:
        # Degenerate case: not enough frequency bins for spectral analysis.
        dominant_frequency_hz = 0.0
        bandpower_0_5_2_hz = 0.0
        bandpower_2_5_hz = 0.0
    else:
        # Find the dominant (peak power) frequency, excluding DC component (index 0).
        positive_frequencies = frequencies[1:]
        positive_power = power[1:]
        if len(positive_power):
            dominant_frequency_hz = float(positive_frequencies[int(np.argmax(positive_power))])
        else:
            dominant_frequency_hz = 0.0

        # Compute power in respiration band (0.5–2 Hz) and higher band (2–5 Hz).
        # Respiration band is typically where respiratory sinus arrhythmia and pulse modulation appear.
        mask_0_5_2 = (frequencies >= 0.5) & (frequencies <= 2.0)
        mask_2_5 = (frequencies >= 2.0) & (frequencies <= 5.0)
        bandpower_0_5_2_hz = (
            float(np.trapezoid(power[mask_0_5_2], frequencies[mask_0_5_2])) if np.any(mask_0_5_2) else 0.0
        )
        bandpower_2_5_hz = (
            float(np.trapezoid(power[mask_2_5], frequencies[mask_2_5])) if np.any(mask_2_5) else 0.0
        )

    # Bandpower ratio indicates spectral shape; useful for distinguishing hypoxemia patterns.
    bandpower_ratio = (
        bandpower_0_5_2_hz / bandpower_2_5_hz if bandpower_2_5_hz > 0.0 else 0.0
    )

    return {
        "mean": mean_value,
        "std": std_value,
        "minimum": minimum_value,
        "maximum": maximum_value,
        "value_range": value_range,
        "rms": rms_value,
        "skewness": skewness_value,
        "kurtosis": kurtosis_value,
        "energy": energy_value,
        "derivative_std": derivative_std_value,
        "dominant_frequency_hz": float(dominant_frequency_hz),
        "bandpower_0_5_2_hz": float(bandpower_0_5_2_hz),
        "bandpower_2_5_hz": float(bandpower_2_5_hz),
        "bandpower_ratio_0_5_2_to_2_5": float(bandpower_ratio),
    }

### Step 3A: Prepare Clean Per-Recording Signals

For each recording, load and validate source files, estimate sampling rate, resample and filter PLETH, and retain aligned SpO2 series for downstream window labeling.

Expected result: one prepared entry per usable recording containing filtered PLETH and clean numerics.

In [ ]:
# Build per-recording prepared inputs before window extraction.
# This cell processes each recording's raw signals and numerics, performs resampling and filtering,
# and prepares clean aligned time series for windowing downstream.
prepared_recordings: list[dict[str, object]] = []

for recording_id in recording_ids:
    signals_path = recording_file_map[recording_id]["Signals"]
    numerics_path = recording_file_map[recording_id]["Numerics"]
    fix_path = recording_file_map[recording_id]["Fix"]

    # ===== LOAD AND VALIDATE FILES =====
    signals_df = pd.read_csv(signals_path, skipinitialspace=True)
    numerics_df = pd.read_csv(numerics_path, skipinitialspace=True)

    # Signals must contain time and PLETH columns
    for required_col in ["Time [s]", "PLETH"]:
        if required_col not in signals_df.columns:
            raise ValueError(f"Missing required column {required_col} in {signals_path}")

    # Numerics must contain time, vital signs, and SpO2
    for required_col in ["Time [s]", "HR", "PULSE", "RESP", "SpO2"]:
        if required_col not in numerics_df.columns:
            raise ValueError(f"Missing required column {required_col} in {numerics_path}")

    # ===== EXTRACT AND CLEAN RAW PLETH SIGNAL =====
    # Convert to numeric arrays and remove rows where either time or PLETH is missing/infinite
    raw_time_seconds = pd.to_numeric(signals_df["Time [s]"], errors="coerce").to_numpy(dtype=float)
    raw_pleth = pd.to_numeric(signals_df["PLETH"], errors="coerce").to_numpy(dtype=float)

    finite_mask = np.isfinite(raw_time_seconds) & np.isfinite(raw_pleth)
    raw_time_seconds = raw_time_seconds[finite_mask]
    raw_pleth = raw_pleth[finite_mask]

    # Skip recordings with insufficient samples
    if len(raw_pleth) < 2:
        continue

    # ===== ESTIMATE SOURCE SAMPLING RATE =====
    # First, try to read the sampling frequency from the .Fix file metadata
    source_rate_hz = None
    with fix_path.open(encoding="utf-8") as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if line.startswith("Signals sampling frequency:"):
                _, value = line.split(":", maxsplit=1)
                source_rate_hz = float(value.replace("Hz", "").strip())
                break

    # Fallback: estimate from median time step between samples
    if source_rate_hz is None:
        step_seconds = np.diff(raw_time_seconds)
        median_step_seconds = float(np.median(step_seconds))
        if not np.isfinite(median_step_seconds) or median_step_seconds <= 0:
            continue
        source_rate_hz = 1.0 / median_step_seconds

    # Skip if estimated rate is invalid
    if not np.isfinite(source_rate_hz) or source_rate_hz <= 0:
        continue

    # Compute recording duration and target sample count for resampling
    duration_seconds = raw_time_seconds[-1] - raw_time_seconds[0]
    if duration_seconds <= 0:
        continue

    # ===== RESAMPLE TO UNIFORM RATE =====
    # Resample raw PLETH from variable source rate to uniform target rate (50 Hz).
    # This ensures all windows have consistent sample counts and enables batch processing.
    target_sample_count = int(round(duration_seconds * target_rate_hz)) + 1
    resampled_pleth = signal.resample(raw_pleth, target_sample_count)

    # ===== APPLY BAND-PASS FILTER =====
    # Design and apply Butterworth band-pass filter (0.5–8 Hz) using second-order sections (sos) for stability.
    # Use causal filtering to avoid future-context leakage from samples that occur later in time.
    sos = signal.butter(
        filter_order,
        [low_cut_hz, high_cut_hz],
        btype="bandpass",
        fs=target_rate_hz,
        output="sos",
    )
    filtered_pleth = signal.sosfilt(sos, resampled_pleth)

    # Skip if filtering produced non-finite values (numerical instability)
    if not np.isfinite(filtered_pleth).all():
        continue

    # ===== PREPARE CLEAN NUMERICS (SpO2 TIMESERIES) =====
    # Extract and align SpO2 time series for window-level labeling.
    # Remove rows with missing time or SpO2 values.
    numeric_time_seconds = pd.to_numeric(numerics_df["Time [s]"], errors="coerce")
    numeric_spo2 = pd.to_numeric(numerics_df["SpO2"], errors="coerce")
    numerics_clean = pd.DataFrame(
        {
            "time_seconds": numeric_time_seconds,
            "spo2": numeric_spo2,
        }
    ).dropna(subset=["time_seconds", "spo2"])

    # Skip if no valid SpO2 samples remain
    if numerics_clean.empty:
        continue

    # ===== STORE PREPARED RECORDING =====
    prepared_recordings.append(
        {
            "recording_id": recording_id,
            "filtered_pleth": filtered_pleth,
            "numerics_clean": numerics_clean,
        }
    )

print(f"Prepared {len(prepared_recordings)} recordings for window extraction")

### Step 3B: Create Sliding Windows And Labels

Iterate through prepared recordings, generate 10s windows at 5s stride, compute median/minimum SpO2 labels, and optionally drop borderline median windows before feature extraction.

Expected result: a feature-row list ready for DataFrame finalization and export.

In [ ]:
# Convert prepared recordings into labeled feature rows.
# For each prepared recording, slide a window across the filtered PLETH signal, compute median and minimum SpO2
# for each window, skip borderline windows (if enabled), extract features, and collect rows.
feature_rows = []
total_candidate_windows = 0
dropped_borderline_windows = 0

for item in prepared_recordings:
    recording_id = str(item["recording_id"])
    filtered_pleth = np.asarray(item["filtered_pleth"], dtype=float)
    numerics_clean = item["numerics_clean"]

    local_window_count = 0

    # ===== SLIDING WINDOW LOOP =====
    # Iterate through fixed-size windows with a fixed stride.
    # Window size: 500 samples (10 seconds at 50 Hz)
    # Stride: 250 samples (5 seconds) - creates 50% overlap between consecutive windows
    for start_index in range(
        0,
        len(filtered_pleth) - window_size_samples + 1,
        stride_size_samples,
    ):
        end_index = start_index + window_size_samples

        # Compute time boundaries for this window (used to query SpO2 measurements)
        start_time_seconds = start_index / target_rate_hz
        end_time_seconds = start_time_seconds + window_seconds

        # ===== QUERY SpO2 FOR WINDOW =====
        # Extract all SpO2 samples that fall within this window's time range.
        # If no SpO2 measurements exist in the window, skip it.
        window_spo2 = numerics_clean.loc[
            (numerics_clean["time_seconds"] >= start_time_seconds)
            & (numerics_clean["time_seconds"] < end_time_seconds),
            "spo2",
        ]

        if window_spo2.empty:
            continue

        total_candidate_windows += 1

        # ===== COMPUTE WINDOW LABELS =====
        # Median SpO2 defines the primary window label for the explainable classification task.
        # Minimum SpO2 is retained as a secondary event-sensitive reference label.
        median_spo2 = float(window_spo2.median())
        min_spo2 = float(window_spo2.min())
        primary_label = int(median_spo2 < low_spo2_threshold)
        label_min_spo2 = int(min_spo2 < low_spo2_threshold)
        is_borderline = borderline_lower <= median_spo2 <= borderline_upper

        if drop_borderline and is_borderline:
            dropped_borderline_windows += 1
            continue

        # ===== EXTRACT PLETH WINDOW =====
        values = filtered_pleth[start_index:end_index]
        if len(values) == 0:
            continue

        # ===== COMPUTE FEATURES =====
        # Call the feature extraction pipeline to compute 14 features from this window.
        window_features = compute_window_features(values, target_rate_hz)

        # ===== APPEND FEATURE ROW =====
        # Store window metadata, primary label, secondary label, and all 14 computed features.
        feature_rows.append(
            {
                "recording_id": recording_id,
                "start_time_seconds": float(start_time_seconds),
                "end_time_seconds": float(end_time_seconds),
                "sample_rate_hz": float(target_rate_hz),
                "median_spo2": float(median_spo2),
                "min_spo2": float(min_spo2),
                "label": int(primary_label),
                "label_min_spo2": int(label_min_spo2),
                **window_features,
            }
        )
        local_window_count += 1

    if recording_id == recording_ids[0]:
        example_windows_count = local_window_count

# ===== WINDOWING SUMMARY =====
kept_windows_count = len(feature_rows)
borderline_drop_pct = (
    (dropped_borderline_windows / total_candidate_windows) * 100.0
    if total_candidate_windows
    else 0.0
)

print(f"Processed {len(prepared_recordings)} prepared recordings")
print(f"Generated {kept_windows_count} feature windows")
print(f"Borderline dropped: {dropped_borderline_windows}/{total_candidate_windows} ({borderline_drop_pct:.2f}%)")

### Step 4: Finalize Feature DataFrame

Convert collected feature rows to a sorted DataFrame and display summary statistics.

In [ ]:
feature_df = pd.DataFrame(feature_rows)
if not feature_df.empty:
    feature_df = feature_df.sort_values(["recording_id", "start_time_seconds"]).reset_index(drop=True)

primary_positive_window_count = int(feature_df["label"].sum()) if not feature_df.empty else 0
secondary_positive_window_count = int(feature_df["label_min_spo2"].sum()) if (not feature_df.empty and "label_min_spo2" in feature_df.columns) else 0
total_window_count = int(len(feature_df))
negative_window_count = total_window_count - primary_positive_window_count
primary_positive_window_pct = (primary_positive_window_count / total_window_count * 100.0) if total_window_count else 0.0
secondary_positive_window_pct = (
    secondary_positive_window_count / total_window_count * 100.0
    if total_window_count
    else 0.0
)

primary_recording_positive = (
    feature_df.groupby("recording_id")["label"].max().astype(int)
    if not feature_df.empty
    else pd.Series(dtype=int)
)
secondary_recording_positive = (
    feature_df.groupby("recording_id")["label_min_spo2"].max().astype(int)
    if not feature_df.empty and "label_min_spo2" in feature_df.columns
    else pd.Series(dtype=int)
)
primary_positive_recording_count = int(primary_recording_positive.sum()) if not primary_recording_positive.empty else 0
total_feature_recordings = int(primary_recording_positive.shape[0])
primary_positive_recording_pct = (
    primary_positive_recording_count / total_feature_recordings * 100.0
    if total_feature_recordings
    else 0.0
)
secondary_positive_recording_count = int(secondary_recording_positive.sum()) if not secondary_recording_positive.empty else 0
secondary_positive_recording_pct = (
    secondary_positive_recording_count / total_feature_recordings * 100.0
    if total_feature_recordings
    else 0.0
)

print(f"Generated {total_window_count} feature windows across {len(recording_ids)} recordings")
print(
    f"Primary median-SpO2 labels: positives={primary_positive_window_count}, negatives={negative_window_count}, "
    f"positive_rate={primary_positive_window_pct:.2f}%"
)
print(
    f"Secondary minimum-SpO2 labels: positives={secondary_positive_window_count}, "
    f"positive_rate={secondary_positive_window_pct:.2f}%"
)
print(
    f"Primary recording-level positives: {primary_positive_recording_count}/{total_feature_recordings} "
    f"({primary_positive_recording_pct:.2f}%)"
)
print(
    f"Secondary recording-level positives: {secondary_positive_recording_count}/{total_feature_recordings} "
    f"({secondary_positive_recording_pct:.2f}%)"
)

feature_df.head()

## Summary Statistics And Export

Report candidate vs kept windows, then write canonical BIDMC artifacts for downstream baseline training in notebook 03.

In [ ]:
# Borderline drop impact summary
borderline_drop_pct = (
    100.0 * dropped_borderline_windows / total_candidate_windows
    if total_candidate_windows > 0
    else 0.0
)
kept_windows_count = len(feature_df)

print(f"Candidate windows with SpO2: {total_candidate_windows}")
print(f"Dropped borderline windows: {dropped_borderline_windows}")
print(f"Borderline drop percentage: {borderline_drop_pct:.2f}%")
print(f"Kept windows for training: {kept_windows_count}")

pd.DataFrame(
    [
        {
            "candidate_windows": int(total_candidate_windows),
            "dropped_borderline_windows": int(dropped_borderline_windows),
            "borderline_drop_pct": float(borderline_drop_pct),
            "kept_windows": int(kept_windows_count),
            "drop_borderline": bool(drop_borderline),
        }
    ]
)

In [ ]:
audit_df.to_csv(AUDIT_CSV, index=False)
feature_df.to_csv(FEATURE_CSV, index=False)

summary = pd.DataFrame(
    [
        {
            "recording_count": int(len(recording_ids)),
            "example_window_count": int(example_windows_count),
            "audit_row_count": int(len(audit_df)),
            "feature_row_count": int(len(feature_df)),
            "artifact_status": "written",
            "audit_csv": str(AUDIT_CSV),
            "feature_csv": str(FEATURE_CSV),
        }
    ]
)
summary